# Generate MATS Linelist from HAPI call

In [1]:
#Import Statements
import numpy as np
import pandas as pd
import os, sys
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
sns.set_style("ticks")
sns.set_context("poster")
import MATS

from MATS.hapi import *
pd.set_option("display.max_rows", 101)

HAPI version: 1.3.0.0
To get the most up-to-date version please check http://hitran.org/hapi
ATTENTION: Python versions of partition sums from TIPS-2021 are now available in HAPI code

           MIT license: Copyright 2021 HITRAN team, see more at http://hitran.org. 

           If you use HAPI in your research or software development,
           please cite it using the following reference:
           R.V. Kochanov, I.E. Gordon, L.S. Rothman, P. Wcislo, C. Hill, J.S. Wilzewski,
           HITRAN Application Programming Interface (HAPI): A comprehensive approach
           to working with spectroscopic data, J. Quant. Spectrosc. Radiat. Transfer 177, 15-30 (2016)
           DOI: 10.1016/j.jqsrt.2016.03.005

           ATTENTION: This is the core version of the HITRAN Application Programming Interface.
                      For more efficient implementation of the absorption coefficient routine, 
                      as well as for new profiles, parameters and other functional,
      

As currently written, the linelists generated by this script will be saved in the HITRAN_to_Dataframe folder.  Alternatively, the os.chdir() function can be used to changing the working directory.

The HAPI print_iso_id function prints the entire molecule directory for HITRAN including the global isotope id numbers that are necessary to interface with HITRANOnline to aquire spectroscopic data.  Additionally, information on the molecule number, local isotope_id, abundance, mass, and molecule name are provided. 

In [2]:

print_iso_id() 

The dictionary "ISO_ID" contains information on "global" IDs of isotopologues in HITRAN

   id            M    I                    iso_name       abundance       mass        mol_name
    1     :      1    1                     H2(16O)    0.9973173000  18.010565             H2O
    2     :      1    2                     H2(18O)    0.0019998270  20.014811             H2O
    3     :      1    3                     H2(17O)    0.0003718841  19.014780             H2O
    4     :      1    4                     HD(16O)    0.0003106928  19.016740             H2O
    5     :      1    5                     HD(18O)    0.0000006230  21.020985             H2O
    6     :      1    6                     HD(17O)    0.0000001159  20.020956             H2O
  129     :      1    7                     D2(16O)    0.0000000242  20.022915             H2O
    7     :      2    1                 (12C)(16O)2    0.9842043000  43.989830             CO2
    8     :      2    2                 (13C)(16O)2    0

The HITRANlinelist_to_csv takes a list of global isotope numbers and minimum and maximum wavenumbers as variables with a tablename, filename, temperature, and option to calculate the speed-dependent broadening as optional parameters.  The spectroscopic data for the isotopes in the global isotope list over the specified wavenumber range will be retrieved from HITRANOnline.  HITRAN breaks-up the HTP line parameters into temperature regimes, so the temperature specied selects for the most relevant parameters.  The tablename parameter sets the ame of the table generated by the HAPI call, where the filename parameter sets the base for the resulting .csv files.  The HITRANlinelist_to_csv function generates two outputs, the first is a HITRAN line list with all data available in HITRAN for the isotopes and spectral range (based on the parsed parameters in the HITRAN_parameter_list).  The second file generates the highest order line shape list using the HITRAN values and formats for MATS.  This line list also will fill in temperature dependences and missing broadener information and if calculate_aw is True calculate the speed dependence based on theory. 



In [3]:







def RETRIEVE_HITRAN_PARAMETERS(global_isotopes, minimum_wavenumber, maximum_wavenumber, perturbers = ['air', 'self'], filename = 'test', 
                   intensity_cutoff = 1e-30, T_range = 296, proxy_perturber = 'air'):
    #T_range options are 50, 150, 296, 700
                   
    

    if proxy_perturber not in perturbers:
        perturbers.extend([proxy_perturber])
    
    other_temps = [str(t) for t in ['50', '150', '296', '700'] if str(t) != str(T_range)]
    other_temp_suffixes = tuple(f"_{t}" for t in other_temps)
    
    parameter_retrieval_list_VP = []
    for param in PARLIST_VOIGT_ALL:
        for perturber in perturbers:
            if perturber in param:
                parameter_retrieval_list_VP.append(param)
    
    parameter_retrieval_list_SDVP = []
    for param in PARLIST_SDVOIGT_ALL:
        for perturber in perturbers:
            if perturber in param:
                parameter_retrieval_list_SDVP.append(param)
    
    
    parameter_retrieval_list_HTP = []
    for param in PARLIST_HT_ALL:
        if param.endswith(other_temp_suffixes):
            pass
        else:
            for perturber in perturbers:
                if perturber in param:
                    parameter_retrieval_list_HTP.append(param)
                    perturber
    
                
    PARLIST_CORE = ['trans_id', 'molec_id','local_iso_id', 'nu','sw','a','elower', 'gp','gpp', 'statep','statepp']
    parameter_retrieval_list_core_VP_SDVP = mergeParlist(PARLIST_CORE, parameter_retrieval_list_VP, parameter_retrieval_list_SDVP)        
        
    
    db_begin('data')
    fetch_by_ids('SDVP', global_isotopes, minimum_wavenumber, maximum_wavenumber, Parameters=parameter_retrieval_list_core_VP_SDVP)
    fetch_by_ids('HTP', global_isotopes, minimum_wavenumber, maximum_wavenumber, Parameters=parameter_retrieval_list_HTP)

    linelist = pd.DataFrame()
    for par_name in parameter_retrieval_list_core_VP_SDVP: # adds standard items to the linelist
            linelist[par_name] = LOCAL_TABLE_CACHE['SDVP']['data'][par_name]
    for par_name in parameter_retrieval_list_HTP: # adds standard items to the linelist
            linelist[par_name] = LOCAL_TABLE_CACHE['HTP']['data'][par_name]
    
    linelist = linelist[linelist['sw'] > intensity_cutoff]
    #Option to save this linelist!
    linelist.to_excel(filename + '_HITRAN.xlsx', index = False)

    return linelist

    
def SINGLE_PERTURBER_MATS_FROM_HITRAN(df, raw_df, perturber, T, proxy_perturber = 'air'):
    """
    Populates standard mHTP parameters utilizing a fallback hierarchy:
    HTP -> SDVP -> VP -> 0.0
    If perturber is empty then defaults to VP for proxy perturber
    """

    #THIS PORTION MAKES A MATS LINELIST.  NOTE THAT IT IS GOING PARAMETER BY PARAMETER IN LS COMPLEXITY NO GUARENTEE FOR A SELF CONSISTENT LINE LIST.
    
    # HT columns
    ht_gamma0   = f'gamma_HT_0_{perturber}_{T}'
    ht_n_gamma0 = f'n_HT_{perturber}_{T}'
    ht_gamma2   = f'gamma_HT_2_{perturber}_{T}'
    ht_n_gamma2 = f'n_gamma_HT_2_{perturber}_{T}'
    ht_delta0   = f'delta_HT_0_{perturber}_{T}'
    ht_n_delta0 = f'deltap_HT_{perturber}_{T}'
    ht_delta2   = f'delta_HT_2_{perturber}_{T}'
    ht_nuOptRe = f'nu_HT_{perturber}'
    ht_n_nuOptRe = f'kappa_HT_{perturber}'
    ht_eta = f'eta_HT_{perturber}'
    ht_y = f'Y_HT_{perturber}_{T}'
    
    
    # SDV columns
    sdv_gamma0   = f'gamma_SDV_0_{perturber}_{T}'
    sdv_n_gamma0 = f'n_SDV_{perturber}_{T}'
    sdv_gamma2   = f'gamma_SDV_2_{perturber}_{T}'
    sdv_n_gamma2 = f'n_gamma_SDV_2_{perturber}_{T}'
    sdv_delta0   = f'delta_SDV_0_{perturber}_{T}'
    sdv_n_delta0 = f'deltap_SDV_{perturber}_{T}'
    sdv_y = f'Y_SDV_{perturber}_{T}'
    sdv_SD_gamma = f'SD_{perturber}'
    
    # VP columns
    vp_gamma0   = f'gamma_{perturber}'
    vp_n_gamma0 = f'n_{perturber}'
    vp_delta0   = f'delta_{perturber}'
    vp_n_delta0 = f'deltap_{perturber}'
    vp_y = f'y_{perturber}'
    
    # Target MATS mHTP columns
    MATS_gamma0   = f'gamma0_{perturber}'
    MATS_n_gamma0 = f'n_gamma0_{perturber}'
    MATS_gamma2   = f'gamma2_{perturber}'
    MATS_SD_gamma = f'SD_gamma_{perturber}'
    MATS_n_gamma2 = f'n_gamma2_{perturber}'
    MATS_delta0   = f'delta0_{perturber}'
    MATS_n_delta0 = f'n_delta0_{perturber}'
    MATS_delta2   = f'delta2_{perturber}'
    MATS_SD_delta = f'SD_delta_{perturber}'
    MATS_n_delta2 = f'n_delta2_{perturber}'
    MATS_nuOptRe   = f'nuOptRe_{perturber}'
    MATS_n_nuOptRe   = f'n_nuOptRe_{perturber}'
    MATS_nuOptIm  = f'nuOptIm_{perturber}'
    MATS_n_nuOptIm   = f'n_nuOptIm_{perturber}'
    MATS_y  = f'y_{perturber}'
    MATS_n_y   = f'n_y_{perturber}'
   
    def fallback(ht_col, sdv_col=None, vp_col = None):
        series = df.get(ht_col, pd.Series(np.nan, index=df.index))
        if sdv_col in df.columns:
            series = series.combine_first(df[sdv_col])
        if vp_col and vp_col in df.columns:
            series = series.combine_first(df[vp_col])

        return series.fillna(0.0)

    #VP
    df[MATS_gamma0]   = fallback(ht_gamma0, sdv_gamma0, vp_gamma0)
    df[MATS_n_gamma0] = fallback(ht_n_gamma0, sdv_n_gamma0, vp_n_gamma0)
    df[MATS_delta0]   = fallback(ht_delta0, sdv_delta0, vp_delta0)
    df[MATS_n_delta0] = fallback(ht_n_delta0, sdv_n_delta0, vp_n_delta0)
    
    #SDVP
    df[MATS_gamma2]   = fallback(ht_gamma2, sdv_gamma2)
    df[MATS_SD_gamma] = df[MATS_gamma2] / df[MATS_gamma0] 
    df[MATS_n_gamma2] = fallback(ht_n_gamma2, sdv_n_gamma2)
    df[MATS_delta2] =  fallback(ht_delta2)
    df[MATS_SD_delta] = df[MATS_delta2] / df[MATS_delta0] 
    df[MATS_n_delta2] = 0
    #HTP
    df[MATS_nuOptRe] = fallback(ht_nuOptRe)
    df[MATS_n_nuOptRe] = fallback(ht_n_nuOptRe)
    df[MATS_nuOptIm] = fallback(ht_eta) # Moving to mHTP where the HITRAN entries are still HTP.  
    df[MATS_n_nuOptIm] = 0
    #Linemixing
    df[MATS_y] = fallback(ht_y, sdv_y, vp_y)
    df[MATS_n_y] = 0

    #Missing Values --> Unphysical

    missing_mask = df[MATS_gamma0] == 0.0
    missing_count = missing_mask.sum()
    if missing_count > 0:
        print(f"Warning: '{perturber}' missing for {missing_count} line(s). Proxying those specific lines to VP '{proxy_perturber}'.")
        def get_proxy_series(raw_col):
            if raw_col in raw_df.columns:
                return raw_df[raw_col]
            else:
                return pd.Series(0.0, index=df.index)
        df.loc[missing_mask, MATS_gamma0]   = get_proxy_series(f'gamma_{proxy_perturber}')[missing_mask]
        df.loc[missing_mask, MATS_n_gamma0] = get_proxy_series(f'n_{proxy_perturber}')[missing_mask]
        df.loc[missing_mask, MATS_delta0]   = get_proxy_series(f'delta_{proxy_perturber}')[missing_mask]
        df.loc[missing_mask, MATS_n_delta0] = get_proxy_series(f'deltap_{proxy_perturber}')[missing_mask]
        df[MATS_n_delta0] = df[MATS_n_delta0].fillna(0)
        
        df.loc[missing_mask, MATS_SD_gamma] = 0.0
        df.loc[missing_mask, MATS_n_gamma2] = 0.0
        df.loc[missing_mask, MATS_SD_delta] = 0.0
        df.loc[missing_mask, MATS_n_delta2] = 0.0
        df.loc[missing_mask, MATS_nuOptRe]  = 0.0
        df.loc[missing_mask, MATS_n_nuOptRe]= 0.0
        df.loc[missing_mask, MATS_nuOptIm]  = 0.0
        df.loc[missing_mask, MATS_n_nuOptIm]= 0.0
        df.loc[missing_mask, MATS_y]= 0.0
        df.loc[missing_mask, MATS_n_y]= 0.0
        

    ##CLEANUP
    source_cols = [
        ht_gamma0, ht_n_gamma0, ht_gamma2, ht_n_gamma2, ht_delta0, ht_n_delta0, ht_delta2, ht_nuOptRe, ht_n_nuOptRe, ht_eta, ht_y,
        sdv_gamma0, sdv_n_gamma0, sdv_gamma2, sdv_n_gamma2, sdv_delta0, sdv_n_delta0, sdv_y,
        vp_gamma0, vp_n_gamma0, vp_delta0, vp_n_delta0, vp_y, sdv_SD_gamma,
        MATS_gamma2, MATS_delta2
    ]

    mats_cols = [
        MATS_gamma0, MATS_n_gamma0, MATS_SD_gamma, MATS_n_gamma2, MATS_delta0, MATS_n_delta0, MATS_SD_delta, MATS_n_delta2,
        MATS_nuOptRe, MATS_n_nuOptRe, MATS_nuOptIm, MATS_n_nuOptIm, MATS_y, MATS_n_y
    ]

    cols_to_drop = [col for col in source_cols if col not in mats_cols]

    df = df.drop(columns=cols_to_drop, errors='ignore')

    ordered_mats_cols = [
        MATS_gamma0, MATS_n_gamma0, 
        MATS_delta0, MATS_n_delta0,
        MATS_SD_gamma, MATS_n_gamma2, 
        MATS_SD_delta, MATS_n_delta2,
        MATS_nuOptRe, MATS_n_nuOptRe, 
        MATS_nuOptIm, MATS_n_nuOptIm, 
        MATS_y, MATS_n_y
    ]

    front_cols = ['trans_id', 'molec_id', 'local_iso_id', 'nu', 'sw', 'elower', 'a']

    column_order = front_cols + ordered_mats_cols

    other_cols = [col for col in df.columns if col not in column_order]

    column_order += other_cols
    
    df = df[column_order]
    


    return df



def MATS_FROM_HITRAN(global_isotopes, minimum_wavenumber, maximum_wavenumber, perturbers = ['air', 'self'], filename = 'test', 
                   intensity_cutoff = 1e-30, T_range = 296, proxy_perturber = 'air'):
    

    HITRAN_linelist = RETRIEVE_HITRAN_PARAMETERS(global_isotopes, minimum_wavenumber, maximum_wavenumber, perturbers = perturbers, filename = filename, 
                       intensity_cutoff = intensity_cutoff, T_range = T_range, proxy_perturber = proxy_perturber)

    MATS_LINELIST = HITRAN_linelist.copy()
    for p in perturbers:
        MATS_LINELIST = SINGLE_PERTURBER_MATS_FROM_HITRAN(MATS_LINELIST,HITRAN_linelist, p, T_range, proxy_perturber = proxy_perturber)

    MATS_LINELIST.to_excel(filename + '_MATS.xlsx', index = False)

    return MATS_LINELIST
    



In [4]:
#Example for O2_air and He in the A-band region
global_isotopes = [36, 37, 38]
minimum_wavenumber = 13000
maximum_wavenumber = 13165
perturbers = ['air', 'He']
base_filename = 'O2 Linelist'

MATS_LINELIST = MATS_FROM_HITRAN(global_isotopes, minimum_wavenumber, maximum_wavenumber, perturbers = perturbers, filename = base_filename, 
                   intensity_cutoff = 1e-30, T_range = 296, proxy_perturber = 'air')




Using data

HTP
                     Lines parsed: 6857
SDVP
                     Lines parsed: 6857

Data is fetched from http://hitran.org

BEGIN DOWNLOAD: SDVP
  65536 bytes written to data/SDVP.data
  65536 bytes written to data/SDVP.data
Header written to data/SDVP.header
END DOWNLOAD
                     Lines parsed: 380
PROCESSED

Data is fetched from http://hitran.org

BEGIN DOWNLOAD: HTP
  65536 bytes written to data/HTP.data
  65536 bytes written to data/HTP.data
Header written to data/HTP.header
END DOWNLOAD
                     Lines parsed: 380
PROCESSED


This example generates a MATS table for CO and the main isotope of CO2 in the 6200 to 6500 wavenumber region.  Additionally, it calculates the theoretical aw value and assumes that measurements are at 296 (temperature = 296 is default).

In [5]:

tablename = 'CO'
global_isotopes = [26, 27, 28, 29, 30,31,7]
wave_min = 6200 
wave_max = 6500 

CO_LINELIST = MATS_FROM_HITRAN(global_isotopes, wave_min, wave_max, perturbers = ['air'], filename = tablename)

Using data

HTP
                     Lines parsed: 380
SDVP
                     Lines parsed: 380

Data is fetched from http://hitran.org

BEGIN DOWNLOAD: SDVP
  65536 bytes written to data/SDVP.data
  65536 bytes written to data/SDVP.data
  65536 bytes written to data/SDVP.data
  65536 bytes written to data/SDVP.data
  65536 bytes written to data/SDVP.data
  65536 bytes written to data/SDVP.data
  65536 bytes written to data/SDVP.data
  65536 bytes written to data/SDVP.data
  65536 bytes written to data/SDVP.data
  65536 bytes written to data/SDVP.data
  65536 bytes written to data/SDVP.data
  65536 bytes written to data/SDVP.data
  65536 bytes written to data/SDVP.data
  65536 bytes written to data/SDVP.data
  65536 bytes written to data/SDVP.data
  65536 bytes written to data/SDVP.data
  65536 bytes written to data/SDVP.data
  65536 bytes written to data/SDVP.data
  65536 bytes written to data/SDVP.data
  65536 bytes written to data/SDVP.data
  65536 bytes written to data/SDVP.data

In [6]:

tablename = 'CO2_30012_Rbranch'
global_isotopes = [7, 8, 9, 10, 11, 12, 13, 14, 15]
wave_min = 6330 
wave_max = 6380 

CO2_LINELIST = MATS_FROM_HITRAN(global_isotopes, wave_min, wave_max, perturbers = ['air'], filename = tablename)

Using data

HTP
                     Lines parsed: 6857
SDVP
                     Lines parsed: 6857

Data is fetched from http://hitran.org

BEGIN DOWNLOAD: SDVP
  65536 bytes written to data/SDVP.data
  65536 bytes written to data/SDVP.data
  65536 bytes written to data/SDVP.data
  65536 bytes written to data/SDVP.data
  65536 bytes written to data/SDVP.data
  65536 bytes written to data/SDVP.data
  65536 bytes written to data/SDVP.data
  65536 bytes written to data/SDVP.data
  65536 bytes written to data/SDVP.data
  65536 bytes written to data/SDVP.data
  65536 bytes written to data/SDVP.data
  65536 bytes written to data/SDVP.data
Header written to data/SDVP.header
END DOWNLOAD
                     Lines parsed: 2252
PROCESSED

Data is fetched from http://hitran.org

BEGIN DOWNLOAD: HTP
  65536 bytes written to data/HTP.data
  65536 bytes written to data/HTP.data
  65536 bytes written to data/HTP.data
  65536 bytes written to data/HTP.data
  65536 bytes written to data/HTP.data
  6